In [3]:
import requests
import pandas as pd

api_key = '47744911c3aa7ef31b9ef77ff9f9a5e7590815db'
year = 2022

# NYC county FIPS codes (Bronx, Brooklyn, Manhattan, Queens, Staten Island)
nyc_counties = ['005', '047', '061', '081', '085']

variables = [
    'NAME',
    'B01003_001E',  # Total population
    'B19013_001E',  # Median household income
    'B02001_002E',  # White population
    'B02001_003E',  # Black population
    'B02001_004E',  # American Indian population
    'B02001_005E',  # Asian population
    'B02001_006E',  # Pacific Islander population
    'B02001_007E',  # Some other race
    'B02001_008E',  # Two or more races
    # Age groups for under 15: male + female for 0-4, 5-9, 10-14
    'B01001_003E',  # Male under 5
    'B01001_004E',  # Male 5-9
    'B01001_005E',  # Male 10-14
    'B01001_027E',  # Female under 5
    'B01001_028E',  # Female 5-9
    'B01001_029E',  # Female 10-14
    # Age groups for over 65: male + female for 65+
    'B01001_020E',  # Male 65-66
    'B01001_021E',  # Male 67-69
    'B01001_022E',  # Male 70-74
    'B01001_023E',  # Male 75-79
    'B01001_024E',  # Male 80-84
    'B01001_025E',  # Male 85+
    'B01001_044E',  # Female 65-66
    'B01001_045E',  # Female 67-69
    'B01001_046E',  # Female 70-74
    'B01001_047E',  # Female 75-79
    'B01001_048E',  # Female 80-84
    'B01001_049E',  # Female 85+
]

state_code = '36'  # New York

url = f'https://api.census.gov/data/{year}/acs/acs5?get={",".join(variables)}&for=tract:*&in=state:{state_code}%20county:{",".join(nyc_counties)}'
response = requests.get(url, params={'key': api_key})

if response.status_code == 200:
    raw = response.json()
    headers = raw[0]
    data = raw[1:]

    rows = []
    for row in data:
        d = dict(zip(headers, row))

        # Convert all numeric fields
        for k in d:
            if k != 'NAME':
                try:
                    d[k] = int(d[k])
                except (ValueError, TypeError):
                    d[k] = None

        pop = d['B01003_001E']

        # Racial composition (proportions)
        white_pct      = d['B02001_002E'] / pop if pop else None
        black_pct      = d['B02001_003E'] / pop if pop else None
        aian_pct       = d['B02001_004E'] / pop if pop else None
        asian_pct      = d['B02001_005E'] / pop if pop else None
        nhpi_pct       = d['B02001_006E'] / pop if pop else None
        other_pct      = d['B02001_007E'] / pop if pop else None
        two_or_more_pct = d['B02001_008E'] / pop if pop else None

        # Under 15
        under_15 = sum(d[v] for v in [
            'B01001_003E', 'B01001_004E', 'B01001_005E',
            'B01001_027E', 'B01001_028E', 'B01001_029E'
        ] if d[v] is not None)
        under_15_pct = under_15 / pop if pop else None

        # Over 65
        over_65 = sum(d[v] for v in [
            'B01001_020E', 'B01001_021E', 'B01001_022E',
            'B01001_023E', 'B01001_024E', 'B01001_025E',
            'B01001_044E', 'B01001_045E', 'B01001_046E',
            'B01001_047E', 'B01001_048E', 'B01001_049E'
        ] if d[v] is not None)
        over_65_pct = over_65 / pop if pop else None

        geoid = f"{d['state']}{d['county']}{d['tract']}"

        rows.append({
            'geoid':                  geoid,
            'tract_name':             d['NAME'],
            'state':                  d['state'],
            'county':                 d['county'],
            'tract':                  d['tract'],
            'total_population':       pop,
            'median_household_income': d['B19013_001E'],
            'white_pct':              white_pct,
            'black_pct':              black_pct,
            'aian_pct':               aian_pct,
            'asian_pct':              asian_pct,
            'nhpi_pct':               nhpi_pct,
            'other_race_pct':         other_pct,
            'two_or_more_races_pct':  two_or_more_pct,
            'under_15_pct':           under_15_pct,
            'over_65_pct':            over_65_pct,
        })

    df = pd.DataFrame(rows)
    df.to_csv('../data/output/census_tracts_acs_2022.csv', index=False)
    print(f"Saved {len(df)} tracts.")
    print(df.head())

else:
    print(f"Error {response.status_code}: {response.text}")

Saved 2327 tracts.
     geoid                                  tract_name  state  county  tract  \
0   365100      Census Tract 1; Bronx County; New York     36       5    100   
1   365200      Census Tract 2; Bronx County; New York     36       5    200   
2   365400      Census Tract 4; Bronx County; New York     36       5    400   
3  3651600     Census Tract 16; Bronx County; New York     36       5   1600   
4  3651901  Census Tract 19.01; Bronx County; New York     36       5   1901   

   total_population  median_household_income  white_pct  black_pct  aian_pct  \
0              4446               -666666666   0.426901   0.464238  0.002024   
1              4870                   115064   0.154004   0.302464  0.000000   
2              6257                   100553   0.126259   0.303180  0.000000   
3              6177                    41362   0.089202   0.371054  0.034483   
4              2181                    49500   0.213663   0.515818  0.015589   

   asian_pct  nhpi_